In [1]:
print("hello python")

hello python


### Data Loading and Preprocessing

In [42]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [43]:
print(torch.cuda.is_available())       # True?
print(torch.cuda.get_device_name(0))   # Should show RTX 4060

True
NVIDIA GeForce RTX 4060 Laptop GPU


In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # should print: cuda

cuda


In [45]:
os.listdir()

['.ipynb_checkpoints', 'Untitled.ipynb']

In [46]:
data = pd.read_csv("../data/league_of_legends_data_large.csv")
data.head()

,win,kills,deaths,assists,gold_earned,cs,wards_placed,wards_killed,damage_dealt
0,0,16,6,19,17088,231,11,7,15367
1,1,8,8,5,14865,259,10,2,38332
2,0,0,17,11,15919,169,14,5,24642
3,0,19,11,1,11534,264,14,3,15789
4,0,12,7,6,18926,124,15,7,40268


In [47]:
#Separate win (target) and the remaining columns (features).
X = data.drop('win',axis=1)
y = data['win']

X_train , X_test , y_train , y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
) 

In [48]:
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [49]:
input_dim = X_train.shape[1]
input_dim

8

In [50]:
X_train.dtype

dtype('float64')

In [51]:
y_test.dtype

dtype('int64')

In [52]:
#Because PyTorch models use float32 by default, and StandardScaler outputs float64
#.values convert pandas series into numpy array
X_train = torch.tensor(X_train,dtype=torch.float32)
y_train = torch.tensor(y_train.values,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)
y_test = torch.tensor(y_test.values,dtype=torch.float32)

In [53]:
X_train.dtype

torch.float32

In [54]:
y_test.dtype

torch.float32

In [55]:
X_train[0].shape

torch.Size([8])

In [56]:
print(X_train.unique())

tensor([-1.8075, -1.7936, -1.7798,  ...,  1.7087,  1.7095,  1.7178])


In [58]:
print(X_test.unique())

tensor([-1.7798, -1.7660, -1.7521, -1.7383, -1.7328, -1.7322, -1.7319, -1.7203,
        -1.7106, -1.7037, -1.6968, -1.6821, -1.6759, -1.6598, -1.6553, -1.6449,
        -1.6414, -1.6386, -1.6329, -1.6315, -1.6276, -1.6268, -1.6252, -1.6239,
        -1.6212, -1.6165, -1.6156, -1.6138, -1.6135, -1.6135, -1.6106, -1.5977,
        -1.5960, -1.5916, -1.5855, -1.5831, -1.5722, -1.5687, -1.5627, -1.5533,
        -1.5524, -1.5443, -1.5408, -1.5301, -1.5169, -1.5115, -1.5098, -1.5031,
        -1.4852, -1.4754, -1.4743, -1.4730, -1.4726, -1.4707, -1.4683, -1.4615,
        -1.4559, -1.4401, -1.4390, -1.4361, -1.4346, -1.4258, -1.4238, -1.4062,
        -1.4017, -1.3947, -1.3891, -1.3828, -1.3785, -1.3719, -1.3564, -1.3370,
        -1.3285, -1.3232, -1.3221, -1.3217, -1.3107, -1.3093, -1.2931, -1.2913,
        -1.2780, -1.2719, -1.2678, -1.2647, -1.2629, -1.2590, -1.2518, -1.2500,
        -1.2401, -1.2366, -1.2265, -1.2263, -1.2176, -1.2165, -1.2142, -1.2119,
        -1.1995, -1.1986, -1.1828, -1.17

In [59]:
print(y_train.unique())

tensor([0., 1.])


In [60]:
print(y_test.unique())

tensor([0., 1.])


### Implement a logistic regression model using PyTorch


In [61]:
class logistic_regression(nn.Module):
    def __init__(self,in_dim):
        super(logistic_regression,self).__init__()
        self.linear = nn.Linear(in_dim,1)
    def forward(self,x):
        return torch.sigmoid(self.linear(x))        

In [62]:
criterion = nn.BCELoss()

In [63]:
# X_train.shape[0] → 800   (samples)
# X_train.shape[1] → 8     (features)
model = logistic_regression(input_dim)

In [64]:
optimizer=optim.SGD(model.parameters(),lr=0.01)

### Train the logistic regression model

In [ ]:
epochs = 1000
y_train = y_train.view(-1, 1) 
data = TensorDataset(X_train,y_train)
trainloader = DataLoader(dataset = data, batch_size = 32)

In [ ]:
print(y_train.unique())

In [ ]:
print(X_train.unique())

In [ ]:
data

In [ ]:
loss1 = 0
for epoch in range(epochs):
    epoch_loss = 0
    for x,y in trainloader:
        yhat = model(x)
        loss = criterion(yhat,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() 

    loss2 = epoch_loss / len(trainloader)
    
    if(epoch+1) % 10 == 0 :

        if abs(loss2 - loss1) < 1e-6:     
            print(f"Early stopping at epoch {epoch+1}")
            break
        else:
            loss1 = loss2
            
        print(f"Epoch {epoch+1} , Loss: {loss2:.4f}")


In [ ]:
model.eval()

with torch.no_grad():
    yhat_train = model(X_train)
    yhat_test = model(X_test)

In [ ]:
train_acc = (yhat_train >=0.5).float().eq(y_train).float().mean()
test_acc = (yhat_test >=0.5).float().eq(y_test.view(-1,1)).float().mean()

In [ ]:
print(f"Train Accuracy: {train_acc.item():.4f}")
print(f"Test Accuracy: {test_acc.item():.4f}")

In [ ]:
print(y_train.unique())   # kya 0 aur 1 dono hain?
print(y_train.dtype)      # float32 hona chahiye
print(X_train.mean(), X_train.std())  # normalized hai?